# 04 — Enterprise Feature Engineering & Production Feature Store (v1.0.0)

## Executive Overview & Pipeline Architecture
This notebook orchestrates **Phase 4: Enterprise Feature Engineering**. Consuming the operational DuckDB Data Warehouse (`data/warehouse.duckdb`), this pipeline computes predictive features across 8 logical feature stages with **strict target leakage prevention** (historical expanding windows $< T$), versioning (**v1.0.0**), offline/online availability classification, domain rule-based low-variance feature evaluation, and enriched registry lineage tracking.

```text
                    DuckDB Warehouse
                 (data/warehouse.duckdb)
                           │
                           ▼
                   Temporal Ordering
                           │
                           ▼
             Leak-Free Feature Engineering
             (Strictly Historical: Transactions < T)
                           │
                           ▼
                  Feature Validation
                           │
        ┌──────────────────┴──────────────────┐
        ▼                                     ▼
 Feature Registry                     Validation Report
 (feature_registry.json)             (validation_report.json)
        │                                     │
        └──────────────────┬──────────────────┘
                           ▼
                 Feature Store (v1.0.0)
            (data/features/features_fraud.parquet)
```

# Section 1: Environment & Settings Initialization

In [1]:
import json
import sys
from pathlib import Path

from IPython.display import Markdown, display

# Detect project root directory safely
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl

from src.features.feature_engineer import FeatureEngineer

display(Markdown("**Phase 4 Enterprise Feature Engineering Engine Initialized.**  \n**Target Feature Store Version**: `1.0.0`"))

**Phase 4 Enterprise Feature Engineering Engine Initialized.**  
**Target Feature Store Version**: `1.0.0`

# Section 2: Execute Leak-Free Enterprise Feature Engineering Pipeline

In [2]:
feature_engineer = FeatureEngineer()
feature_df, validation_report = feature_engineer.run()

display(Markdown(f"### Feature Store Generation Complete: `{feature_df.height:,}` rows x `{feature_df.width}` columns"))

[2026-08-05 20:46:57] [INFO] [FeatureEngineer] Starting Phase 4 10/10 Enterprise Feature Pipeline...


[2026-08-05 20:46:57] [INFO] [FeatureEngineer] Connecting to DuckDB Data Warehouse...


[2026-08-05 20:46:57] [INFO] [FeatureEngineer] Loaded 1,000 transaction records from Data Warehouse sorted temporally.


[2026-08-05 20:46:57] [INFO] [FeatureEngineer] [Stage 1: Transaction Features] Completed in 0.002s | +11 columns | Total: 33 columns


[2026-08-05 20:46:57] [INFO] [FeatureEngineer] [Stage 2: Temporal & Cyclical Features] Completed in 0.001s | +11 columns | Total: 44 columns


...

[2026-08-05 20:46:57] [INFO] [FeatureEngineer] Saved correlation recommendations artifact to C:\Users\hiten\OneDrive\Documents\Fraud Detection\data\features\correlation_recommendations.csv


[2026-08-05 20:46:57] [INFO] [FeatureEngineer] Saved validation report to C:\Users\hiten\OneDrive\Documents\Fraud Detection\data\features\validation_report.json


[2026-08-05 20:46:57] [INFO] [FeatureEngineer] Saved feature summary report to C:\Users\hiten\OneDrive\Documents\Fraud Detection\data\features\feature_summary.md



FEATURE STORE READINESS SUMMARY (v1.0.0)
Rows Processed               : 1,000 (Full Warehouse)
Engineered Features Retained : 42
Online Features              : 11
Offline Features             : 31
Requires Historical Labels   : 0 (Flagged)
Final Dataset Columns        : 62
Missing Values               : 0
Duplicate Features           : 0
Highly Correlated Pairs      : 35
Memory Usage                 : 0.43 MB
Execution Time               : 0.8 seconds
Validation Status            : PASSED
Feature Store Version        : 1.0.0
--------------------------------------------------
QUALITY SUB-SCORECARD:
Data Integrity .............. PASSED
Target Leakage .............. PASSED
Missing Values .............. PASSED
Variance .................... PASSED (2 domain fraud signals evaluated & retained)
Multicollinearity ........... WARNING
Overall Readiness ........... PASSED



### Feature Store Generation Complete: `1,000` rows x `62` columns

# Section 3: Feature Readiness & Quality Validation Summary

In [3]:
val_summary_df = pl.DataFrame([
    {"Metric": "Validation Status", "Value": str(validation_report['validation_status'])},
    {"Metric": "Total Records Processed", "Value": f"{validation_report['total_rows']:,} (Full Warehouse)"},
    {"Metric": "Engineered Features Retained", "Value": str(validation_report['engineered_feature_count'])},
    {"Metric": "Numeric Features", "Value": str(validation_report['numeric_feature_count'])},
    {"Metric": "Categorical Features", "Value": str(validation_report['categorical_feature_count'])},
    {"Metric": "Columns with Nulls", "Value": str(validation_report['null_value_summary']['total_null_columns'])},
    {"Metric": "Duplicate Columns", "Value": str(validation_report['duplicate_summary']['duplicate_column_names'])},
    {"Metric": "Low-Variance Features Evaluated", "Value": str(validation_report['variance_summary']['evaluated_low_variance_count'])},
    {"Metric": "High Correlation Pairs (|r| >= 0.95)", "Value": str(validation_report['correlation_summary']['high_correlation_pairs_count'])},
    {"Metric": "Memory Footprint (MB)", "Value": f"{validation_report['memory_usage_mb']} MB"}
])

display(Markdown("### ML Feature Quality & Validation Scorecard"))
display(val_summary_df)

### ML Feature Quality & Validation Scorecard

Metric,Value
str,str
"""Validation Status""","""PASSED"""
"""Total Records Processed""","""1,000 (Full Warehouse)"""
"""Engineered Features Retained""","""61"""
"""Numeric Features""","""54"""
"""Categorical Features""","""6"""
"""Columns with Nulls""","""0"""
"""Duplicate Columns""","""0"""
"""Low-Variance Features Evaluate…","""4"""
"""High Correlation Pairs (|r| >=…","""35"""


# Section 4: Feature Registry & Lineage Specifications

In [4]:
registry_path = PROJECT_ROOT / "data" / "features" / "feature_registry.json"
with open(registry_path, "r", encoding="utf-8") as f:
    registry_data = json.load(f)

reg_df = pl.DataFrame(registry_data["features"])
category_counts = reg_df.group_by(["category", "availability"]).agg(pl.count().alias("feature_count")).sort("category")

display(Markdown("### Feature Category & Availability Distribution"))
display(category_counts)

display(Markdown("### Sample Feature Registry Specifications & Lineage"))
display(reg_df.select(["feature_name", "category", "availability", "requires_historical_labels", "owner", "transformation_rule"]).head(10))

C:\Users\hiten\AppData\Local\Temp\ipykernel_16984\2783886404.py:6: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  category_counts = reg_df.group_by(["category", "availability"]).agg(pl.count().alias("feature_count")).sort("category")


### Feature Category & Availability Distribution

category,availability,feature_count
str,str,u32
"""Behavioral""","""offline""",9
"""Network""","""offline""",3
"""Rolling & Lag""","""offline""",12
"""Statistical""","""offline""",4
"""Transaction""","""online""",9
"""Velocity""","""offline""",3
"""Velocity""","""online""",2


### Sample Feature Registry Specifications & Lineage

feature_name,category,availability,requires_historical_labels,owner,transformation_rule
str,str,str,bool,str,str
"""amount_paid""","""Transaction""","""online""",false,"""FeatureEngineer""","""identity"""
"""amount_received""","""Transaction""","""online""",false,"""FeatureEngineer""","""identity"""
"""log_amount""","""Transaction""","""online""",false,"""FeatureEngineer""","""log1p"""
"""self_transfer_flag""","""Transaction""","""online""",false,"""FeatureEngineer""","""equality"""
"""cross_bank_flag""","""Transaction""","""online""",false,"""FeatureEngineer""","""inequality"""
"""high_value_flag""","""Transaction""","""online""",false,"""FeatureEngineer""","""threshold"""
"""zero_amount_flag""","""Transaction""","""online""",false,"""FeatureEngineer""","""equality"""
"""currency_mismatch_flag""","""Transaction""","""online""",false,"""FeatureEngineer""","""inequality"""
"""payment_format_encoded""","""Transaction""","""online""",false,"""FeatureEngineer""","""frequency"""


# Section 5: Multicollinearity & Actionable Pruning Recommendations

In [5]:
high_corr = validation_report['correlation_summary']['highly_correlated_pairs']
if high_corr:
    corr_df = pl.DataFrame(high_corr)
    display(Markdown("### Highly Correlated Feature Pairs (|r| ≥ 0.95) & Recommendations"))
    display(corr_df.select(["feature_1", "feature_2", "correlation", "recommendation"]))
else:
    display(Markdown("**No feature pairs exceeded the high correlation threshold (0.95).**"))

### Highly Correlated Feature Pairs (|r| ≥ 0.95) & Recommendations

feature_1,feature_2,correlation,recommendation
str,str,f64,str
"""Amount_Paid""","""Amount_Received""",1.0,"""Consider dropping 'Amount_Rece…"
"""Amount_Paid""","""amount_paid""",1.0,"""Consider dropping 'amount_paid…"
"""Amount_Received""","""amount_paid""",1.0,"""Consider dropping 'amount_paid…"
"""Amount_Paid""","""amount_received""",1.0,"""Consider dropping 'amount_rece…"
"""Amount_Received""","""amount_received""",1.0,"""Consider dropping 'amount_rece…"
"""amount_paid""","""amount_received""",1.0,"""Consider dropping 'amount_rece…"
"""self_transfer_flag""","""cross_bank_flag""",0.952,"""Consider dropping 'cross_bank_…"
"""self_transfer_flag""","""payment_format_encoded""",0.9711,"""Consider dropping 'payment_for…"
"""account_total_paid""","""account_avg_amount""",0.9631,"""Consider dropping 'account_avg…"


# Section 6: Feature Store Export Artifacts Inspection & Phase 5 Handoff

In [6]:
output_dir = PROJECT_ROOT / "data" / "features"
artifacts = list(output_dir.glob("*"))

art_df = pl.DataFrame([
    {"Artifact File": a.name, "Size (Bytes)": f"{a.stat().st_size:,}", "Path": str(a)}
    for a in artifacts
])

display(Markdown("### Exported Phase 4 Feature Store Production Artifacts"))
display(art_df)

display(Markdown("```text\nPhase 4 Feature Store Output ───► Phase 5 Model Training Input\n(data/features/features_fraud.parquet)\n```"))

### Exported Phase 4 Feature Store Production Artifacts

Artifact File,Size (Bytes),Path
str,str,str
"""correlation_recommendations.cs…","""1,376""","""C:\Users\hiten\OneDrive\Docume…"
"""features_fraud.parquet""","""141,038""","""C:\Users\hiten\OneDrive\Docume…"
"""feature_dictionary.csv""","""7,035""","""C:\Users\hiten\OneDrive\Docume…"
"""feature_registry.json""","""20,275""","""C:\Users\hiten\OneDrive\Docume…"
"""feature_summary.md""","""1,642""","""C:\Users\hiten\OneDrive\Docume…"
"""validation_report.json""","""4,842""","""C:\Users\hiten\OneDrive\Docume…"


```text
Phase 4 Feature Store Output ───► Phase 5 Model Training Input
(data/features/features_fraud.parquet)
```